In [219]:
from hmp_utils import split_amass_dataset, print_amass_sets_info, load_sets_by_dataset_dict

In [2]:
sets_by_dataset_dict=split_amass_dataset("path/to/datasets/AMASS","path/to/output/amass_split")

In [3]:
data=load_sets_by_dataset_dict("path/to/output/amass_split")
print_amass_sets_info(data)

Getting number of frames in ACCAD/train:  30%|███       | 46/151 [00:00<00:00, 457.04it/s]

Getting number of frames in EKUT/validation: 100%|██████████| 15/15 [00:00<00:00, 610.16it/s]
Getting number of frames in EyesJapanDataset/Eyes_Japan_Dataset/train: 100%|██████████| 534/534 [00:02<00:00, 186.01it/s]
Getting number of frames in EyesJapanDataset/Eyes_Japan_Dataset/test: 100%|██████████| 114/114 [00:00<00:00, 203.82it/s]
Getting number of frames in EyesJapanDataset/Eyes_Japan_Dataset/validation: 100%|██████████| 102/102 [00:00<00:00, 202.19it/s]
Getting number of frames in TotalCapture/test: 100%|██████████| 12/12 [00:00<00:00, 187.69it/s]
Getting number of frames in TotalCapture/validation: 100%|██████████| 10/10 [00:00<00:00, 168.80it/s]
Getting number of frames in Transitions/test: 100%|██████████| 6/6 [00:00<00:00, 445.45it/s]
Getting number of frames in Transitions/validation: 100%|██████████| 6/6 [00:00<00:00, 435.91it/s]

Dataset name: AMASS
  Number of mocaps: 7558
  Number of frames: 12919838
  Total number of minutes: 1794.422
  Min number of frames: 360
  Max number of frames: 22948
  Mean number of frames: 1709.426

  Set name: train
    Number of mocaps: 6389
    Percentage of mocaps: 84.533%
    Number of frames: 10407803
    Total number of minutes: 1445.528
    Percentage of frames: 80.557%
    Min number of frames: 360
    Max number of frames: 22948
    Mean number of frames: 1629.019
  Set name: test
    Number of mocaps: 578
    Percentage of mocaps: 7.648%
    Number of frames: 1209931
    Total number of minutes: 168.046
    Percentage of frames: 9.365%
    Min number of frames: 360
    Max number of frames: 17799
    Mean number of frames: 2093.306
  Set name: validation
    Number of mocaps: 591
    Percentage of mocaps: 7.820%
    Number of frames: 1302104
    Total number of minutes: 180.848
    Percentage of frames: 10.078%
    Min number of frames: 360
    Max number of frames: 1870

Dataset name: AMASS
  Number of mocaps: 8775
  Number of frames: 13252539
  Total number of minutes: 1840.630
  Min number of frames: 150
  Max number of frames: 22948
  Mean number of frames: 1510.261

  Set name: train
    Number of mocaps: 7474
    Percentage of mocaps: 85.174%
    Number of frames: 10689666
    Total number of minutes: 1484.676
    Percentage of frames: 80.661%
    Min number of frames: 150
    Max number of frames: 22948
    Mean number of frames: 1430.247
  Set name: test
    Number of mocaps: 644
    Percentage of mocaps: 7.339%
    Number of frames: 1221525
    Total number of minutes: 169.656
    Percentage of frames: 9.217%
    Min number of frames: 153
    Max number of frames: 14762
    Mean number of frames: 1896.778
  Set name: validation
    Number of mocaps: 657
    Percentage of mocaps: 7.487%
    Number of frames: 1341348
    Total number of minutes: 186.298
    Percentage of frames: 10.121%
    Min number of frames: 164
    Max number of frames: 16137
    Mean number of frames: 2041.626

In [8]:
# Results with MIN_MOCAP_FRAMES = TARGET_FRAMERATE * 6  # 6 seconds as in SRNN


total_number_frames=11381588
total_number_mocaps=4532

valid_number_frames=1221410
valid_number_mocaps=417

f_for_mocap=120*3 #(75=25*3)
seq_for_mocap=3

selected_valid_number_frames=valid_number_mocaps*f_for_mocap*seq_for_mocap
print("selected_valid_number_frames",selected_valid_number_frames)

print("Percentage of frames: ",valid_number_frames/total_number_frames*100)
print("Percentage of valid frames: ",selected_valid_number_frames/total_number_frames*100)
print("Minutes of mocaps: ",valid_number_frames/120/60)
print("Minutes of valid mocaps: ",selected_valid_number_frames/120/60)

selected_valid_number_frames 450360
Percentage of frames:  10.731455048276215
Percentage of valid frames:  3.9569170839780883
Minutes of mocaps:  169.64027777777775
Minutes of valid mocaps:  62.55


In [5]:
import numpy as np
import math
from typing import Union

AMASS_EVALUATION_SEED = 130297
AMASS_INPUT_SEQ_LEN = 2  # in seconds
AMASS_OUTPUT_SEQ_LEN = 1  # in seconds
AMASS_SEQ_LEN = AMASS_INPUT_SEQ_LEN + AMASS_OUTPUT_SEQ_LEN  # in seconds
AMASS_MAX_SEQUENCES_BY_MOCAP = 3


def get_start_indices_amass(mocap_len: int, seq_len: int) -> list:
    """
    Find start frames indices for standard evaluation with AMASS dataset.
    ---
    ### Args:
    - `mocap_len` (int): number of frames of the mocap.
    - `seq_len` (int): length of the sequence in frames.
    """
    rng = np.random.RandomState(AMASS_EVALUATION_SEED)

    subsections = min(mocap_len // seq_len, AMASS_MAX_SEQUENCES_BY_MOCAP)

    last_index = mocap_len - seq_len + 1  # first not usable index in the mocap
    if last_index > AMASS_MAX_SEQUENCES_BY_MOCAP * seq_len:
        first_index = math.ceil(seq_len / 2)
        last_index -= math.floor(seq_len / 2)
    elif mocap_len % seq_len > math.floor(seq_len / 2):
        first_index = math.floor(seq_len / 2)
    elif subsections > 1:
        first_index = math.ceil(seq_len / 2)
        subsections -= 1
    else:
        first_index = math.ceil((mocap_len % seq_len) / 2)

    if subsections > 0:
        subseq_len = (last_index - first_index) // subsections


    frame_idxs = []
    for idx in range(subsections):

        frame_idxs.append(
            rng.randint(
                first_index + idx * subseq_len, first_index + (idx + 1) * subseq_len
            )
        )

    return frame_idxs


def get_amass_evaluation_frames(
    num_frames_list: Union[list, np.ndarray],
    input_len: int,
    output_len: int,
    framerate: int,
) -> list:
    """
    Returns a list of lists (num_mocap, num_seq) of frame indices selected for evaluation.
    ---
    ### Args:
    - `num_frames_list` (list or np.ndarray): list of number of frames of the mocaps.
    - `input_len` (int): number of input frames.
    - `output_len` (int): number of output frames.
    - `framerate` (int): framerate of the mocaps.
    """
    if input_len > AMASS_INPUT_SEQ_LEN * framerate:
        raise ValueError(
            "input_len must be smaller than or equal to AMASS_INPUT_SEQ_LEN(2s) * framerate"
        )
    if output_len > AMASS_OUTPUT_SEQ_LEN * framerate:
        raise ValueError(
            "output_len must be smaller than or equal to AMASS_OUTPUT_SEQ_LEN(1s) * framerate"
        )

    frame_idxs_list = []
    for num_frames in num_frames_list:
        frame_idxs = []
        for start_idx in get_start_indices_amass(
            mocap_len=num_frames, seq_len=AMASS_SEQ_LEN * framerate
        ):
            t0 = start_idx + AMASS_INPUT_SEQ_LEN * framerate
            frame_idxs.append(range(t0 - input_len, t0 + output_len))
        frame_idxs_list.append(frame_idxs)

    return frame_idxs_list

In [218]:
h=get_amass_evaluation_frames(num_frames_list=[120,15], input_len=20, output_len=10, framerate=10)
print(h)

[[range(34, 64), range(38, 68), range(58, 88)], []]


In [3]:
from hmp_utils import split_amass_dataset, print_amass_sets_info, load_sets_by_dataset_dict, MocapLoader, BodyModelType

def get_num_frames(mocap_path: str, mocap_loader: MocapLoader) -> int:
    """
    Returns the number of frames in a mocap file.
    ---
    ### Args:
    - `mocap_path` (str): path to the mocap file.
    """
    mocap = mocap_loader.load_mocap(mocap_path)
    return mocap.trans.shape[0]  # trans: translation of root joint (num_frames, 1, 3)

mocap_loader = MocapLoader(
        body_model_type=BodyModelType.SMPL,
        keypoint_ids_to_load=[0],
        target_framerate=120,
    )

data=load_sets_by_dataset_dict("path/to/output/amass_split")

all_valid_mocaps=[]
for mocaps in data['validation'].values():
    for mocap in mocaps:
        all_valid_mocaps.append(mocap)

print(len(all_valid_mocaps))
len_all_valid_mocaps=[get_num_frames(mocap_path, mocap_loader) for mocap_path in all_valid_mocaps]


591


In [6]:
h=get_amass_evaluation_frames(num_frames_list=len_all_valid_mocaps, input_len=240, output_len=120, framerate=120)
print(h)

[[range(62, 422)], [range(269, 629)], [range(115, 475)], [range(48, 408)], [range(295, 655)], [range(10, 370)], [range(16, 376)], [range(269, 629), range(660, 1020)], [range(18, 378)], [range(269, 629)], [range(295, 655)], [range(48, 408)], [range(295, 655), range(460, 820)], [range(269, 629)], [range(269, 629)], [range(48, 408)], [range(60, 420)], [range(1, 361)], [range(295, 655), range(468, 828)], [range(112, 472)], [range(114, 474)], [range(295, 655)], [range(70, 430)], [range(295, 655), range(516, 876)], [range(295, 655)], [range(295, 655), range(512, 872), range(861, 1221)], [range(295, 655)], [range(269, 629), range(635, 995), range(941, 1301)], [range(199, 559)], [range(295, 655), range(496, 856)], [range(269, 629)], [range(269, 629), range(718, 1078)], [range(199, 559)], [range(199, 559)], [range(295, 655)], [range(58, 418)], [range(48, 408)], [range(66, 426)], [range(269, 629), range(694, 1054)], [range(269, 629), range(694, 1054)], [range(110, 470)], [range(269, 629)], [rang

In [9]:
tot=0
for idx,mocap in enumerate(all_valid_mocaps):
    mocap = mocap_loader.load_mocap(mocap)
    for idxs in h[idx]:
        #print(mocap.trans[idxs].shape)
        tot+=1
print(tot)
print(len(all_valid_mocaps)*3)

1233
1773


In [1]:
1233*3/60

61.65